# Agentic AI Systems — Worklet 15
# Executive Decision Support System
## Lab 6 · OpenCLA / OpenClaw Runtime — skills, tools, memory, audit

**Phase A · Foundations** &nbsp;|&nbsp; **Goal:** run the agent inside a controlled runtime with a full audit log.

---

### What Lab 6 contributes to Worklet 15

| | |
|---|---|
| **Inputs** | Skills (Lab 3), connector (Lab 5), memory (Lab 4). |
| **Outputs** | A governed run with audit + checkpoints. |
| **Agents / nodes** | Coordinator hardened into the **runtime controller**. |
| **Deliverable** | Runtime wrapping the agent: budgets, gates, checkpoint/resume, audit log. |
| **Acceptance** | **Every action is logged with inputs/outputs; a crashed run resumes from its last checkpoint.** |

### The idea

Labs 3–5 each added a capability. None of them controls *the run*. A runtime is the thing the loop executes **inside**, and it owns four things the agent is not allowed to own itself:

| | |
|---|---|
| **budgets** | a run cannot spend more actions, connector calls or seconds than it was granted |
| **audit log** | every action, with its inputs, its output, its duration — and its failures |
| **checkpoints** | state written to disk after every completed action, so a crash resumes |
| **gates** | actions that stop and wait for a **named** human |

Everything goes through one method, `perform()`. That is why the audit log is complete *by construction* rather than by discipline.

### What this notebook actually demonstrates

1. **Cell 5 — a real crash.** The benchmark backend drops mid-run at `compute_trend`, three actions in. The failure is written to the audit log *and* checkpointed.
2. **Cell 6 — the acceptance criterion.** A brand-new `Runtime` object is rebuilt from the file on disk. The three completed actions are **skipped, not redone**; the run continues and finishes.
3. **Cell 7 — four refusals.** Action budget, connector budget, dependency ordering, and a human gate that cannot be walked past.

### How to run

Top to bottom, **no API key and no network**. Cell 1 writes `edss_core.py` (Labs 1–2) and `edss_stack.py` (Labs 3–5), so this notebook is standalone. Cell 8 executes the rubric as assertions.

---

### Cell 1 · The accumulated stack

Two files are written to disk so this notebook runs standalone:

* **`edss_core.py`** — Labs 1–2: institutional data, typed tools, `ToolRouter`, validation gate.
* **`edss_stack.py`** — Labs 3–5: the `Skill` framework and two skills, `RunState`/`Archive` with tag + TF-IDF retrieval, and the MCP server over file + SQLite + API.

*(Collapse both — prior work, restated only for self-containment.)*

In [1]:
%%writefile edss_core.py
"""edss_core — shared foundation for the Worklet 15 lab notebooks.

Worklet 15 · Executive Decision Support System · Agentic AI (26ECAC401)

This is the Lab 1 + Lab 2 foundation, unchanged: the institutional data, the
typed tool contracts, the single ToolRouter choke point, the Connector, and the
validation gate. Labs 3-8 build on top of it rather than restating it, so every
notebook works against the same tools and the same numbers.

Written to disk by the first code cell of each lab notebook, so each notebook
runs standalone in Colab with no setup.
"""

from __future__ import annotations

import csv
import io
import json
import math
import re
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

# 1 · Institutional data (identical to Lab 2, Cell 3)
# =============================================================================

REPORTING_PERIOD = "academic year 2025-26"

DEPARTMENT_REPORTS_CSV = """dept_code,dept_name,category,students_appeared,students_passed,co_po_attainment,sanctioned_faculty,publications,citations,grants_lakh,eligible_students,placed_students,total_package_lpa,interns_completed,budget_allocated_lakh,budget_spent_lakh,figure_status
CSE,Computer Science & Engineering,circuit,240,231,2.71,34,58,402,96.5,228,212,2247.2,205,420,393,final
ECE,Electronics & Communication,circuit,180,168,2.58,26,34,221,41.0,172,143,1058.2,149,310,276,final
ISE,Information Science & Engineering,circuit,120,116,2.66,18,21,140,22.5,114,103,947.6,96,240,221,final
EEE,Electrical & Electronics,circuit,120,109,2.41,17,16,88,18.0,116,86,524.6,84,210,181,final
MECH,Mechanical Engineering,core,180,158,2.33,24,19,95,27.5,174,111,599.4,102,300,247,final
CIVIL,Civil Engineering,core,120,101,2.19,16,11,47,12.0,118,62,285.2,61,190,149,final
CHEM,Chemical Engineering,core,60,54,2.28,11,14,76,15.5,58,33,168.3,34,140,118,final
BT,Biotechnology,core,60,55,2.35,10,12,63,9.5,57,29,142.1,30,130,102,final
"""

NUMERIC_COLUMNS = {
    "students_appeared": int, "students_passed": int, "co_po_attainment": float,
    "sanctioned_faculty": int, "publications": int, "citations": int,
    "grants_lakh": float, "eligible_students": int, "placed_students": int,
    "total_package_lpa": float, "interns_completed": int,
    "budget_allocated_lakh": float, "budget_spent_lakh": float,
}

KPI_DEFINITIONS: Dict[str, Dict[str, Any]] = {
    "KPI-A1": {"name": "Institution pass percentage", "domain": "Academic",
               "method": "ratio", "numerator": "students_passed",
               "denominator": "students_appeared", "scale": 100, "unit": "%",
               "formula": "sum(students_passed) / sum(students_appeared) x 100",
               "target": 92.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Academic ERP"},
    "KPI-A2": {"name": "CO/PO attainment index", "domain": "Academic",
               "method": "weighted_mean", "value_field": "co_po_attainment",
               "weight_field": "students_appeared", "unit": "index 0-3",
               "formula": "sum(co_po_attainment x students_appeared) / sum(students_appeared)",
               "target": 2.50, "target_source": "NAAC criterion 2",
               "direction": "higher_is_better", "source_system": "Academic ERP (OBE module)"},
    "KPI-R1": {"name": "Publications per faculty", "domain": "Research",
               "method": "ratio", "numerator": "publications",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(publications) / sum(sanctioned_faculty)",
               "target": 1.50, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R2": {"name": "Citations per faculty", "domain": "Research",
               "method": "ratio", "numerator": "citations",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(citations) / sum(sanctioned_faculty)",
               "target": 9.00, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R3": {"name": "Grants sanctioned", "domain": "Research",
               "method": "sum", "value_field": "grants_lakh", "unit": "INR lakh",
               "formula": "sum(grants_lakh)",
               "target": 300.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-P1": {"name": "Placement percentage", "domain": "Placement",
               "method": "ratio", "numerator": "placed_students",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(placed_students) / sum(eligible_students) x 100",
               "target": 80.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P2": {"name": "Average package", "domain": "Placement",
               "method": "ratio", "numerator": "total_package_lpa",
               "denominator": "placed_students", "scale": 1, "unit": "LPA",
               "formula": "sum(total_package_lpa) / sum(placed_students)",
               "target": 8.00, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P3": {"name": "Internship completion", "domain": "Placement",
               "method": "ratio", "numerator": "interns_completed",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(interns_completed) / sum(eligible_students) x 100",
               "target": 85.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-F1": {"name": "Budget utilisation", "domain": "Finance",
               "method": "ratio", "numerator": "budget_spent_lakh",
               "denominator": "budget_allocated_lakh", "scale": 100, "unit": "%",
               "formula": "sum(budget_spent_lakh) / sum(budget_allocated_lakh) x 100",
               "target": 90.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Finance & Accounts ledger",
               "figure_status": "provisional",
               "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes "
                         "books in August."},
}

BENCHMARK_API_DB = {
    "KPI-R1": {"peer_median": 1.82, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-R2": {"peer_median": 11.40, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-P1": {"peer_median": 78.60, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
}

DOMAINS = ["Academic", "Research", "Placement", "Finance"]
DOMAIN_KPIS = {d: [k for k, v in KPI_DEFINITIONS.items() if v["domain"] == d] for d in DOMAINS}

# Registrar-authorised read access, per Gate 1 of the hybrid execution methodology.
ACCESS_REGISTER = {
    "Academic":  {"source_system": "Academic ERP — examination & OBE records",
                  "read_access": "granted by Registrar on 2026-06-01", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["pass percentage", "CO/PO attainment", "curriculum compliance"]},
    "Research":  {"source_system": "Research & Publications repository (Scopus-synced)",
                  "read_access": "granted by Registrar on 2026-06-01", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["publications", "citations", "grants sanctioned"]},
    "Placement": {"source_system": "Placement Cell tracker",
                  "read_access": "granted by Registrar on 2026-06-04", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["placement percentage", "average package",
                              "internship completion"],
                  "caveat": "3 provisional offers are excluded until the Placement Cell "
                            "verifies them."},
    "Finance":   {"source_system": "Finance & Accounts ledger",
                  "read_access": "granted by Registrar on 2026-06-07",
                  "figure_status": "provisional",
                  # 2025-26 is deliberately absent: the books are not closed, and the
                  # Lab 1 planner must notice that rather than plan over it.
                  "periods": ["2023-24", "2024-25"],
                  "metrics": ["budget utilisation", "revenue", "grant funds"],
                  "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes "
                            "books in August."},
}


# =============================================================================
# 2 · Tool contracts (identical to Lab 2, Cell 4)
# =============================================================================

TOOL_SCHEMAS: Dict[str, Dict[str, Any]] = {
    "read_department_reports": {
        "description": "Read per-department metric rows for one reporting period.",
        "parameters": {"type": "object", "properties": {
            "period": {"type": "string"},
            "departments": {"type": "array", "items": {"type": "string"}},
            "category": {"type": "string", "enum": ["circuit", "core"]}},
            "required": ["period"]},
        "returns": "{rows, row_count, source_system} | {error}"},
    "read_kpi_definitions": {
        "description": "Read approved KPI definitions: formula, unit, target, authority.",
        "parameters": {"type": "object", "properties": {
            "kpi_ids": {"type": "array", "items": {"type": "string"}}}, "required": []},
        "returns": "{definitions, count} | {error}"},
    "read_external_benchmark": {
        "description": "Published peer median for a KPI. Coverage is partial.",
        "parameters": {"type": "object", "properties": {"kpi_id": {"type": "string"}},
                       "required": ["kpi_id"]},
        "returns": "{kpi_id, peer_median, cohort, year} | {error}"},
    "filter_rows": {
        "description": "Filter department rows by an exact field value; errors on empty.",
        "parameters": {"type": "object", "properties": {
            "rows": {"type": "array", "items": {"type": "object"}},
            "field": {"type": "string"}, "value": {"type": "string"}},
            "required": ["rows", "field", "value"]},
        "returns": "{rows, row_count, filter} | {error}"},
    "compute_kpi": {
        "description": "Compute one approved KPI over department rows, with evidence.",
        "parameters": {"type": "object", "properties": {
            "kpi_id": {"type": "string"},
            "rows": {"type": "array", "items": {"type": "object"}}},
            "required": ["kpi_id", "rows"]},
        "returns": "{kpi_id, value, unit, formula, evidence, source_system} | {error}"},
    "compare_to_target": {
        "description": "Compare a KPI value to its approved target; flags rounding risk.",
        "parameters": {"type": "object", "properties": {
            "kpi_id": {"type": "string"}, "value": {"type": "number"}},
            "required": ["kpi_id", "value"]},
        "returns": "{value, target, variance, status, rounding_warning} | {error}"},
    "compute_variance": {
        "description": "Difference between a KPI value and any reference figure.",
        "parameters": {"type": "object", "properties": {
            "value": {"type": "number"}, "reference": {"type": "number"},
            "label": {"type": "string"}}, "required": ["value", "reference", "label"]},
        "returns": "{variance, abs_variance, pct_of_reference} | {error}"},
    "draft_notification": {
        "description": "DRAFT a leadership notification. Never sends. Sources required.",
        "parameters": {"type": "object", "properties": {
            "audience": {"type": "string"}, "subject": {"type": "string"},
            "findings": {"type": "array", "items": {"type": "object"}}},
            "required": ["audience", "subject", "findings"]},
        "returns": "{status: 'DRAFT - NOT SENT', body, requires_approval_from} | {error}"},
}


def _typed_rows(text: str) -> List[Dict[str, Any]]:
    rows = []
    for raw in csv.DictReader(io.StringIO(text)):
        row = dict(raw)
        for col, caster in NUMERIC_COLUMNS.items():
            if col in row:
                try:
                    row[col] = caster(row[col])
                except (TypeError, ValueError):
                    return [{"error": f"column '{col}' is not {caster.__name__}"}]
        rows.append(row)
    return rows


# Optional demo latency, so the parallel/sequential distinction in the run
# timeline is visible. Off by default; never affects any computed figure.
SOURCE_LATENCY_SECONDS = 0.0


def read_department_reports(period: str, departments=None, category=None) -> Dict[str, Any]:
    if period != REPORTING_PERIOD:
        return {"error": f"no department report export for '{period}'; the only loaded "
                         f"period is '{REPORTING_PERIOD}'"}
    if SOURCE_LATENCY_SECONDS:
        time.sleep(SOURCE_LATENCY_SECONDS)
    rows = _typed_rows(DEPARTMENT_REPORTS_CSV)
    if rows and "error" in rows[0]:
        return {"error": rows[0]["error"]}
    if departments:
        known = {r["dept_code"] for r in rows}
        unknown = [d for d in departments if d not in known]
        if unknown:
            return {"error": f"unknown department code(s): {unknown}; known: {sorted(known)}"}
        rows = [r for r in rows if r["dept_code"] in departments]
    if category:
        rows = [r for r in rows if r["category"] == category]
        if not rows:
            return {"error": f"no departments in category '{category}'"}
    if not rows:
        return {"error": "empty department report export"}
    return {"rows": rows, "row_count": len(rows), "period": period,
            "source_system": "Department reports export (Registrar-authorised read)"}


def read_kpi_definitions(kpi_ids=None) -> Dict[str, Any]:
    if kpi_ids:
        unknown = [k for k in kpi_ids if k not in KPI_DEFINITIONS]
        if unknown:
            return {"error": f"unknown KPI id(s): {unknown}"}
        defs = {k: KPI_DEFINITIONS[k] for k in kpi_ids}
    else:
        defs = dict(KPI_DEFINITIONS)
    return {"definitions": defs, "count": len(defs),
            "source_system": "Institutional KPI register (approved definitions)"}


def read_external_benchmark(kpi_id: str) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    record = BENCHMARK_API_DB.get(kpi_id)
    if record is None:
        return {"error": f"no published external benchmark for {kpi_id} "
                         f"({KPI_DEFINITIONS[kpi_id]['name']}) in the benchmark API"}
    return {"kpi_id": kpi_id, **record, "source_system": "External benchmark API (NIRF cohort)"}


def filter_rows(rows, field: str, value: str) -> Dict[str, Any]:
    if not rows:
        return {"error": "filter_rows received an empty row set"}
    if field not in rows[0]:
        return {"error": f"unknown field '{field}'; available: {sorted(rows[0])}"}
    subset = [r for r in rows if str(r[field]) == str(value)]
    if not subset:
        return {"error": f"filter {field}={value!r} matched 0 of {len(rows)} rows — "
                         f"refusing to return an empty subset"}
    return {"rows": subset, "row_count": len(subset), "filter": f"{field}=={value}",
            "filtered_from": len(rows)}


def compute_kpi(kpi_id: str, rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Generic: the arithmetic comes from the KPI definition, not from this code."""
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    if not rows:
        return {"error": f"cannot compute {kpi_id} over an empty row set"}
    d = KPI_DEFINITIONS[kpi_id]
    method, evidence = d["method"], []
    try:
        if method == "ratio":
            num_f, den_f, scale = d["numerator"], d["denominator"], d["scale"]
            num = sum(r[num_f] for r in rows)
            den = sum(r[den_f] for r in rows)
            if den == 0:
                return {"error": f"{kpi_id}: denominator sum({den_f}) is zero"}
            value = num / den * scale
            inputs = {f"sum({num_f})": round(num, 4), f"sum({den_f})": round(den, 4)}
            for r in rows:
                dv = (r[num_f] / r[den_f] * scale) if r[den_f] else None
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[num_f], "denominator": r[den_f],
                                 "numerator_field": num_f, "denominator_field": den_f,
                                 "dept_value": round(dv, 2) if dv is not None else None,
                                 "share_of_denominator_pct": round(r[den_f] / den * 100, 2)})
        elif method == "weighted_mean":
            vf, wf = d["value_field"], d["weight_field"]
            wsum = sum(r[wf] for r in rows)
            if wsum == 0:
                return {"error": f"{kpi_id}: total weight sum({wf}) is zero"}
            value = sum(r[vf] * r[wf] for r in rows) / wsum
            inputs = {f"sum({vf} x {wf})": round(sum(r[vf] * r[wf] for r in rows), 4),
                      f"sum({wf})": wsum}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[vf], "denominator": r[wf],
                                 "numerator_field": vf, "denominator_field": wf,
                                 "dept_value": r[vf],
                                 "share_of_denominator_pct": round(r[wf] / wsum * 100, 2)})
        elif method == "sum":
            vf = d["value_field"]
            value = sum(r[vf] for r in rows)
            inputs = {f"sum({vf})": round(value, 4)}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[vf], "denominator": None,
                                 "numerator_field": vf, "denominator_field": None,
                                 "dept_value": r[vf],
                                 "share_of_denominator_pct": round(r[vf] / value * 100, 2)
                                 if value else None})
        else:
            return {"error": f"unsupported KPI method '{method}' for {kpi_id}"}
    except KeyError as missing:
        return {"error": f"{kpi_id}: department rows are missing column {missing}"}

    statuses = {r.get("figure_status", "final") for r in rows}
    figure_status = d.get("figure_status") or ("final" if statuses == {"final"} else "mixed")
    out = {"kpi_id": kpi_id, "name": d["name"], "domain": d["domain"],
           "value": round(value, 4), "unit": d["unit"], "formula": d["formula"],
           "method": method, "n_departments": len(rows),
           "departments": [r["dept_code"] for r in rows],
           "inputs_used": inputs, "evidence": evidence,
           "source_system": d["source_system"], "figure_status": figure_status}
    if d.get("caveat"):
        out["caveat"] = d["caveat"]
    return out


def compare_to_target(kpi_id: str, value: float) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    d = KPI_DEFINITIONS[kpi_id]
    target = d["target"]
    variance = value - target
    pct_of_target = value / target * 100 if target else None
    gap_pct = abs(variance) / target * 100 if target else 0
    status = "on target" if variance >= 0 else ("at risk" if gap_pct <= 2.0 else "below target")
    rounding_warning = None
    if variance < 0:
        for places in (0, 1):
            if round(value, places) >= target:
                rounding_warning = (f"{value:.4g} rounds to {round(value, places)} at "
                                    f"{places} decimal place(s), which would read as meeting "
                                    f"the target of {target}. Report the unrounded figure.")
                break
    return {"kpi_id": kpi_id, "name": d["name"], "value": round(value, 4), "target": target,
            "target_source": d["target_source"], "unit": d["unit"],
            "variance": round(variance, 4),
            "pct_of_target": round(pct_of_target, 2) if pct_of_target else None,
            "status": status, "rounding_warning": rounding_warning}


def compute_variance(value: float, reference: float, label: str) -> Dict[str, Any]:
    if reference == 0:
        return {"error": f"cannot compute variance against a zero reference ({label})"}
    variance = value - reference
    return {"value": round(value, 4), "reference": round(reference, 4), "label": label,
            "variance": round(variance, 4), "abs_variance": round(abs(variance), 4),
            "pct_of_reference": round(value / reference * 100, 2)}


def draft_notification(audience: str, subject: str, findings: List[Dict[str, Any]]):
    if not findings:
        return {"error": "refusing to draft a notification with no findings"}
    unsourced = [f for f in findings if not f.get("source")]
    if unsourced:
        return {"error": f"{len(unsourced)} finding(s) carry no source; every figure in a "
                         f"leadership notification must cite where it came from"}
    body = "\n".join(f"  {i}. {f['statement']}\n     source: {f['source']}"
                     for i, f in enumerate(findings, 1))
    return {"status": "DRAFT - NOT SENT", "audience": audience, "subject": subject,
            "body": body, "findings_cited": len(findings),
            "requires_approval_from": "Principal / Registrar (Chapter 15 human gate)",
            "note": "This tool cannot send. Circulation happens only after human approval."}


TOOLS = {
    "read_department_reports": read_department_reports,
    "read_kpi_definitions": read_kpi_definitions,
    "read_external_benchmark": read_external_benchmark,
    "filter_rows": filter_rows,
    "compute_kpi": compute_kpi,
    "compare_to_target": compare_to_target,
    "compute_variance": compute_variance,
    "draft_notification": draft_notification,
}


# =============================================================================
# 3 · Router, connector, validation gate (Lab 2, Cells 5-6)
# =============================================================================

class ToolContractError(Exception):
    """Raised before a call when arguments do not satisfy the tool schema."""


class ToolError(Exception):
    """Raised after a call when the tool returned an explicit error."""


_JSON_TYPES = {"string": str, "number": (int, float), "integer": int,
               "boolean": bool, "array": list, "object": dict}


def _summarise(obj, _depth=0):
    if isinstance(obj, list):
        if len(obj) > 3 and all(isinstance(x, dict) for x in obj):
            return f"[{len(obj)} rows: {', '.join(str(x.get('dept_code', '?')) for x in obj)}]"
        return [_summarise(x, _depth + 1) for x in obj]
    if isinstance(obj, dict):
        return {k: _summarise(v, _depth + 1) for k, v in obj.items()}
    if isinstance(obj, str) and len(obj) > 160:
        return obj[:157] + "..."
    return obj


class ToolRouter:
    """The single choke point. Validates, executes, logs, raises on error."""

    def __init__(self, agent: str = "Coordinator"):
        self.agent = agent
        self.trace: List[Dict[str, Any]] = []
        self.raw_outputs: List[Any] = []

    def validate_input(self, tool: str, kwargs: Dict[str, Any]) -> None:
        if tool not in TOOL_SCHEMAS:
            raise ToolContractError(f"no such tool: '{tool}'")
        schema = TOOL_SCHEMAS[tool]["parameters"]
        props, required = schema["properties"], schema.get("required", [])
        for name in required:
            if name not in kwargs:
                raise ToolContractError(f"{tool}: missing required argument '{name}'")
        for name, val in kwargs.items():
            if name not in props:
                raise ToolContractError(
                    f"{tool}: unknown argument '{name}'; accepts {sorted(props)}")
            spec = props[name]
            expected = _JSON_TYPES.get(spec["type"])
            if expected and not isinstance(val, expected):
                raise ToolContractError(
                    f"{tool}.{name}: expected {spec['type']}, got {type(val).__name__}")
            if spec.get("enum") and val not in spec["enum"]:
                raise ToolContractError(f"{tool}.{name}: {val!r} not in {spec['enum']}")

    def call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        self.validate_input(tool, kwargs)
        started = time.perf_counter()
        result = TOOLS[tool](**kwargs)
        entry = {"step": len(self.trace) + 1, "agent": self.agent, "tool": tool,
                 "reason": reason, "input": _summarise(kwargs), "output": _summarise(result),
                 "ok": "error" not in result,
                 "ms": round((time.perf_counter() - started) * 1000, 2),
                 "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        self.trace.append(entry)
        self.raw_outputs.append(result)
        if "error" in result:
            raise ToolError(f"{tool} failed: {result['error']}")
        return result

    def try_call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        try:
            return self.call(tool, reason, **kwargs)
        except ToolError:
            return dict(TOOLS[tool](**kwargs))


class Connector:
    """One governed surface over every source. No agent touches a source directly."""

    RESOURCES = {
        "department reports":              "read_department_reports",
        "institutional KPIs & dashboards": "read_kpi_definitions",
        "external benchmarks":             "read_external_benchmark",
    }

    def __init__(self, router: ToolRouter):
        self.router = router

    def read(self, resource: str, reason: str, required: bool = True, **params):
        if resource not in self.RESOURCES:
            raise ToolContractError(f"connector exposes no resource '{resource}'; "
                                    f"available: {sorted(self.RESOURCES)}")
        caller = self.router.call if required else self.router.try_call
        return caller(self.RESOURCES[resource], reason, **params)


class Validated:
    def __init__(self, checks: List[tuple], subject: str):
        self.checks, self.subject = checks, subject
        self.failing = [name for name, ok, _ in checks if not ok]
        self.valid = not self.failing

    def __repr__(self):
        return f"<Validated {self.subject}: {'VALID' if self.valid else self.failing}>"


def validate_kpi_result(computed: Dict[str, Any],
                        compared: Optional[Dict[str, Any]] = None) -> Validated:
    ev = computed.get("evidence", [])
    checks = [
        ("value is a finite number",
         isinstance(computed.get("value"), (int, float)) and math.isfinite(computed["value"]),
         f"value={computed.get('value')}"),
        ("the formula used is recorded", bool(computed.get("formula")),
         computed.get("formula", "")),
        ("the figure cites its source system", bool(computed.get("source_system")),
         computed.get("source_system", "")),
        ("raw inputs are recorded", bool(computed.get("inputs_used")),
         str(computed.get("inputs_used", ""))),
        ("every department in the evidence is identified",
         bool(ev) and all(e.get("dept_code") for e in ev), f"{len(ev)} evidence rows"),
        ("evidence covers exactly the departments computed over",
         len(ev) == computed.get("n_departments"), f"{len(ev)} rows"),
        ("provisional figures carry a caveat",
         computed.get("figure_status") == "final" or bool(computed.get("caveat")),
         f"figure_status={computed.get('figure_status')}"),
    ]
    if compared is not None:
        checks.append(("the target names the authority that set it",
                       bool(compared.get("target_source")), compared.get("target_source", "")))
    return Validated(checks, computed.get("kpi_id", "?"))


# =============================================================================


Overwriting edss_core.py


In [2]:
%%writefile edss_stack.py
"""edss_stack — the capabilities Labs 3, 4 and 5 added, in one importable layer.

Worklet 15 · Executive Decision Support System · Agentic AI (26ECAC401)

    Lab 3  Skill            reusable, schema-typed capability packages
    Lab 4  RunState/Archive short-term run state vs the long-term baseline,
                            with tag and TF-IDF-similarity retrieval
    Lab 5  MCPServer/Client one typed, scoped connector over file + SQLite + API

Labs 6-8 bind these together rather than restating them: Lab 6 puts a runtime
around them, Lab 7 wires them into a node graph, Lab 8 runs them in parallel.

Written to disk by the first code cells of Labs 6-8 so each notebook is
standalone.
"""

import csv
import io
import json
import math
import re
import sqlite3
import textwrap
import time
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

from edss_core import (DOMAINS, KPI_DEFINITIONS, REPORTING_PERIOD, ACCESS_REGISTER,
                       DEPARTMENT_REPORTS_CSV, NUMERIC_COLUMNS, BENCHMARK_API_DB,
                       ToolRouter, Connector, ToolError, ToolContractError,
                       compute_kpi, compare_to_target, validate_kpi_result,
                       _typed_rows)

_JSON_TYPES = {"string": str, "number": (int, float), "integer": int,
               "boolean": bool, "array": list, "object": dict}


# ==========================================================================
# LAB 3 — the Skill framework
# ==========================================================================

class SkillContractError(Exception):
    """Raised when a skill's input or output violates its declared schema."""


@dataclass
class Skill:
    """A reusable capability package.

    name          stable identifier
    description   what it does, in one line
    instructions  the procedure it follows — the part a human or a model reads
    constraints   what it must never do
    input_schema  typed contract, validated BEFORE the skill runs
    output_schema typed contract, validated AFTER it runs
    tools         the tool names it is permitted to call
    """
    name: str
    description: str
    instructions: str
    constraints: List[str]
    input_schema: Dict[str, Any]
    output_schema: Dict[str, Any]
    tools: List[str]
    _fn: Optional[Callable] = None
    invocations: List[Dict[str, Any]] = field(default_factory=list)

    # ---- schema validation, both directions ----------------------------
    def _validate(self, payload: Dict[str, Any], schema: Dict[str, Any],
                  where: str) -> None:
        props, required = schema.get("properties", {}), schema.get("required", [])
        for name in required:
            if name not in payload:
                raise SkillContractError(
                    f"{self.name} {where}: missing required field '{name}'")
        for name, value in payload.items():
            if name not in props:
                raise SkillContractError(
                    f"{self.name} {where}: unexpected field '{name}'; "
                    f"accepts {sorted(props)}")
            expected = _JSON_TYPES.get(props[name]["type"])
            if expected and not isinstance(value, expected):
                raise SkillContractError(
                    f"{self.name} {where}.{name}: expected {props[name]['type']}, "
                    f"got {type(value).__name__}")
            if props[name].get("enum") and value not in props[name]["enum"]:
                raise SkillContractError(
                    f"{self.name} {where}.{name}: {value!r} not in "
                    f"{props[name]['enum']}")

    def run(self, router: "ToolRouter", **inputs) -> Dict[str, Any]:
        self._validate(inputs, self.input_schema, "input")
        before = len(router.trace)
        result = self._fn(router, **inputs)
        self._validate(result, self.output_schema, "output")
        used = [e["tool"] for e in router.trace[before:]]
        stray = [t for t in used if t not in self.tools]
        if stray:
            raise SkillContractError(
                f"{self.name} called tools outside its declared surface: {stray}")
        self.invocations.append({"inputs": inputs, "tools_used": used,
                                 "output_keys": sorted(result)})
        return result

    def card(self) -> str:
        return (f"SKILL  {self.name}\n"
                f"  {self.description}\n"
                f"  inputs   : {list(self.input_schema.get('properties', {}))}"
                f"  (required: {self.input_schema.get('required', [])})\n"
                f"  outputs  : {list(self.output_schema.get('properties', {}))}\n"
                f"  tools    : {self.tools}\n"
                f"  constraints:\n"
                + "\n".join(f"    - {c}" for c in self.constraints))


SKILL_REGISTRY: Dict[str, Skill] = {}


def register(skill: Skill) -> Skill:
    SKILL_REGISTRY[skill.name] = skill
    return skill


# ==========================================================================
# LAB 3 — skill: review_domain
# ==========================================================================

REVIEW_DOMAIN = Skill(
    name="review_domain",
    description="Review one domain's approved KPIs against their targets, with "
                "per-department evidence.",
    instructions=(
        "1. Read the department rows for the requested period through the connector.\n"
        "2. If a department filter is supplied, narrow to it and refuse an empty subset.\n"
        "3. For every approved KPI belonging to the domain: read its definition, compute\n"
        "   it with the registered formula, and compare it to its approved target.\n"
        "4. Return the KPI set with variance, status, source system and figure status.\n"
        "5. Report which KPIs are off target; never recommend an action — that is the\n"
        "   Report Generation Agent's job, behind the human gate."),
    constraints=[
        "Use only approved KPI definitions; never invent a formula.",
        "Never average department percentages — aggregate raw counts.",
        "Label any figure that is not final; never present it as an actual.",
        "Emit no recommendation and take no action.",
    ],
    input_schema={
        "type": "object",
        "properties": {
            "domain": {"type": "string", "enum": DOMAINS},
            "period": {"type": "string"},
            "category": {"type": "string", "enum": ["circuit", "core"]},
        },
        "required": ["domain", "period"],
    },
    output_schema={
        "type": "object",
        "properties": {
            "domain": {"type": "string"},
            "period": {"type": "string"},
            "scope": {"type": "string"},
            "kpis": {"type": "array"},
            "off_target": {"type": "array"},
            "headline": {"type": "string"},
            "provisional": {"type": "boolean"},
        },
        "required": ["domain", "period", "scope", "kpis", "off_target", "headline",
                     "provisional"],
    },
    tools=["read_department_reports", "filter_rows", "read_kpi_definitions",
           "compute_kpi", "compare_to_target"],
)


def _review_domain(router, domain, period, category=None):
    connector = Connector(router)
    rows = connector.read("department reports",
                          reason=f"review_domain({domain}): per-department metrics",
                          period=period)["rows"]
    scope = "all departments"
    if category:
        rows = router.call("filter_rows",
                           reason=f"review_domain({domain}): narrow to {category} branches",
                           rows=rows, field="category", value=category)["rows"]
        scope = f"{category} departments"

    kpi_ids = [k for k, d in KPI_DEFINITIONS.items() if d["domain"] == domain]
    kpis, provisional = [], False
    for kpi_id in kpi_ids:
        connector.read("institutional KPIs & dashboards",
                       reason=f"{kpi_id}: approved formula and target",
                       kpi_ids=[kpi_id])
        computed = router.call("compute_kpi", reason=f"{kpi_id}: apply the formula",
                               kpi_id=kpi_id, rows=rows)
        compared = router.call("compare_to_target", reason=f"{kpi_id}: variance vs target",
                               kpi_id=kpi_id, value=computed["value"])
        gate = validate_kpi_result(computed, compared)
        if not gate.valid:
            raise ToolError(f"validation gate failed for {kpi_id}: {gate.failing}")
        provisional = provisional or computed["figure_status"] != "final"
        kpis.append({
            "kpi_id": kpi_id, "name": computed["name"], "value": computed["value"],
            "unit": computed["unit"], "target": compared["target"],
            "target_source": compared["target_source"], "variance": compared["variance"],
            "status": compared["status"], "formula": computed["formula"],
            "source_system": computed["source_system"],
            "figure_status": computed["figure_status"],
            "rounding_warning": compared["rounding_warning"],
            "evidence": computed["evidence"],
        })

    off = [k["kpi_id"] for k in kpis if k["status"] != "on target"]
    headline = (f"{domain}: {len(off)} of {len(kpis)} KPIs off target across {scope} "
                f"for {period}." + (" Figures include a provisional source."
                                    if provisional else ""))
    return {"domain": domain, "period": period, "scope": scope, "kpis": kpis,
            "off_target": off, "headline": headline, "provisional": provisional}


REVIEW_DOMAIN._fn = _review_domain
register(REVIEW_DOMAIN)


# ==========================================================================
# LAB 3 — skill: format_document
# ==========================================================================

FORMAT_DOCUMENT = Skill(
    name="format_document",
    description="Render a titled, sectioned payload as a leadership-ready document "
                "in a chosen template.",
    instructions=(
        "1. Emit the title, then the metadata block as key: value lines.\n"
        "2. Emit each section in order: a heading, then its bullets, then its table\n"
        "   if it has one.\n"
        "3. In 'brief' template, number the sections; in 'memo' template, do not.\n"
        "4. Append the source line verbatim if one is supplied.\n"
        "5. Reproduce every value exactly as given — never reformat a number."),
    constraints=[
        "Never compute, round or reword a value; this skill only lays out.",
        "Never drop a section or a caveat that was passed in.",
        "Stay domain-agnostic: no KPI, department or worklet logic here.",
    ],
    input_schema={
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "meta": {"type": "object"},
            "sections": {"type": "array"},
            "template": {"type": "string", "enum": ["brief", "memo"]},
            "source_line": {"type": "string"},
        },
        "required": ["title", "sections"],
    },
    output_schema={
        "type": "object",
        "properties": {"document": {"type": "string"},
                       "template": {"type": "string"},
                       "section_count": {"type": "integer"},
                       "line_count": {"type": "integer"}},
        "required": ["document", "template", "section_count", "line_count"],
    },
    tools=[],          # a pure formatting skill: it calls no tool at all
)


def _format_document(router, title, sections, meta=None, template="brief",
                     source_line=None):
    out = [title, "=" * len(title)]
    for k, v in (meta or {}).items():
        out.append(f"{k}: {v}")
    out.append("")
    for i, sec in enumerate(sections, 1):
        heading = sec.get("heading", "")
        out.append(f"{i}. {heading}" if template == "brief" else heading.upper())
        out.append("-" * 70)
        for b in sec.get("bullets", []):
            out.append(textwrap.fill(f"  - {b}", 88, subsequent_indent="    "))
        table = sec.get("table")
        if table:
            widths = [max(len(str(r[c])) for r in [dict(zip(table["columns"],
                                                            table["columns"]))] + table["rows"])
                      for c in table["columns"]]
            out.append("  " + "  ".join(str(c).ljust(w) for c, w in
                                        zip(table["columns"], widths)))
            for row in table["rows"]:
                out.append("  " + "  ".join(str(row[c]).ljust(w) for c, w in
                                            zip(table["columns"], widths)))
        out.append("")
    if source_line:
        out.append(source_line)
    document = "\n".join(out)
    return {"document": document, "template": template,
            "section_count": len(sections), "line_count": len(document.splitlines())}


FORMAT_DOCUMENT._fn = _format_document
register(FORMAT_DOCUMENT)


# ==========================================================================
# LAB 4 — the memory split
# ==========================================================================

class ArchiveImmutableError(Exception):
    """Raised on any attempt to edit a record that is already archived."""


@dataclass
class RunState:
    """Short-term working memory. One run. Thrown away at the end."""
    run_id: str
    period: str
    draft: Dict[str, Any] = field(default_factory=dict)
    scratch: Dict[str, Any] = field(default_factory=dict)
    retrieved: List[Dict[str, Any]] = field(default_factory=list)

    def note(self, key: str, value: Any) -> None:
        self.scratch[key] = value

    def summary(self) -> str:
        return (f"RunState({self.run_id}) draft_keys={sorted(self.draft)} "
                f"scratch_keys={sorted(self.scratch)} retrieved={len(self.retrieved)}")


class Archive:
    """Long-term store. Append-only; every write is versioned and timestamped."""

    def __init__(self):
        self._records: Dict[str, Dict[str, Any]] = {}
        self.writes: List[Dict[str, Any]] = []

    # ---- writing -------------------------------------------------------
    def put(self, record_id: str, record: Dict[str, Any]) -> Dict[str, Any]:
        if record_id in self._records:
            raise ArchiveImmutableError(
                f"'{record_id}' is already archived; the archive is append-only. "
                f"Write a new version id instead of editing the baseline.")
        stored = {**record, "record_id": record_id,
                  "archived_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        self._records[record_id] = stored
        self.writes.append({"record_id": record_id, "tags": record.get("tags", []),
                            "at": stored["archived_at"]})
        return stored

    # ---- reading -------------------------------------------------------
    def get(self, record_id: str) -> Dict[str, Any]:
        # A COPY, so a caller mutating what it read cannot corrupt the baseline.
        return json.loads(json.dumps(self._records[record_id]))

    def all(self) -> List[Dict[str, Any]]:
        return [self.get(rid) for rid in self._records]

    def __len__(self):
        return len(self._records)

    def __contains__(self, record_id):
        return record_id in self._records


# ==========================================================================
# LAB 4 — what is in the archive
# ==========================================================================

archive = Archive()

PRIOR_BRIEFS = [
    {"record_id": "BRIEF-2023-24-Q4",
     "type": "decision_brief", "period": "2023-24",
     "title": "Placement recovery after the 2023 hiring slowdown",
     "tags": ["placement", "core-branches", "recruiter-outreach", "below-target"],
     "summary": "Placement closed at 71.80% against an 80% target. Core branches drove "
                "the gap; circuit branches held. The Council approved a recruiter "
                "outreach drive and deferred the package target.",
     "decisions": ["Approved INR 12 lakh recruiter outreach budget",
                   "Deferred the average-package target by one year"],
     "outcome": "Placement rose to 73.50% the following year — short of target but "
                "the trend reversed. The wider recruiter pool brought in lower-paying "
                "offers, and average package began to slide."},
    {"record_id": "BRIEF-2024-25-Q2",
     "type": "decision_brief", "period": "2024-25",
     "title": "Research output against the NIRF submission",
     "tags": ["research", "publications", "citations", "nirf", "below-target"],
     "summary": "Publications per faculty stood at 1.05 against a 1.50 target and "
                "citations per faculty at 6.40. The Council approved a seed-grant "
                "scheme and asked for a per-department breakdown each quarter.",
     "decisions": ["Approved a seed-grant scheme for research-active faculty",
                   "Made department-level drill-down mandatory for research KPIs"],
     "outcome": "Publications rose to 1.12; the reporting change stuck and is now "
                "standing practice."},
    {"record_id": "BRIEF-2024-25-Q4",
     "type": "decision_brief", "period": "2024-25",
     "title": "Budget utilisation and the unreconciled Q4 position",
     "tags": ["finance", "budget-utilisation", "provisional", "reconciliation"],
     "summary": "Utilisation was reported at 88.10% but Q4 was unreconciled at the time "
                "of review. The Council declined to act on a provisional figure and "
                "deferred the decision to the post-audit cycle.",
     "decisions": ["Deferred all budget reallocation until the books closed",
                   "Required provisional figures to be labelled on the face of the brief"],
     "outcome": "Final utilisation came in at 86.90% — 1.2 points below the provisional "
                "figure, vindicating the deferral."},
    {"record_id": "BRIEF-2024-25-Q3",
     "type": "decision_brief", "period": "2024-25",
     "title": "Pass percentage and the rounding dispute",
     "tags": ["academic", "pass-percentage", "rounding", "at-risk"],
     "summary": "Pass percentage was circulated as '92%' when the unrounded figure was "
                "91.60%, which is below the 92.00% target. The Council directed that "
                "KPI figures be reported unrounded.",
     "decisions": ["Directed that all KPI figures be reported unrounded",
                   "Ordered remedial coaching in the two weakest departments"],
     "outcome": "The reporting rule is now enforced by the KPI engine itself."},
]

PRIOR_KPI_SNAPSHOTS = [
    {"record_id": "KPI-SNAPSHOT-2023-24", "type": "kpi_snapshot", "period": "2023-24",
     "title": "Institution KPI snapshot 2023-24",
     "tags": ["kpi-snapshot", "closed-year", "baseline"],
     "values": {"KPI-A1": 90.40, "KPI-A2": 2.38, "KPI-R1": 0.98, "KPI-R2": 5.90,
                "KPI-R3": 198.0, "KPI-P1": 71.80, "KPI-P2": 7.10, "KPI-P3": 64.20,
                "KPI-F1": 84.30}},
    {"record_id": "KPI-SNAPSHOT-2024-25", "type": "kpi_snapshot", "period": "2024-25",
     "title": "Institution KPI snapshot 2024-25",
     "tags": ["kpi-snapshot", "closed-year", "baseline"],
     "values": {"KPI-A1": 91.20, "KPI-A2": 2.44, "KPI-R1": 1.12, "KPI-R2": 6.40,
                "KPI-R3": 221.5, "KPI-P1": 73.50, "KPI-P2": 7.95, "KPI-P3": 69.10,
                "KPI-F1": 86.90}},
]

for rec in PRIOR_BRIEFS + PRIOR_KPI_SNAPSHOTS:
    archive.put(rec["record_id"], {k: v for k, v in rec.items() if k != "record_id"})


# ==========================================================================
# LAB 4 — retrieval by tag
# ==========================================================================

def retrieve_by_tags(store: Archive, tags: List[str], mode: str = "any",
                     record_type: Optional[str] = None) -> List[Dict[str, Any]]:
    """mode='any' -> union; mode='all' -> records carrying every tag."""
    if mode not in ("any", "all"):
        raise ValueError("mode must be 'any' or 'all'")
    wanted = {t.lower() for t in tags}
    hits = []
    for rec in store.all():
        if record_type and rec.get("type") != record_type:
            continue
        have = {t.lower() for t in rec.get("tags", [])}
        overlap = wanted & have
        if (mode == "any" and overlap) or (mode == "all" and wanted <= have):
            hits.append({**rec, "matched_tags": sorted(overlap),
                         "tag_score": len(overlap) / len(wanted)})
    return sorted(hits, key=lambda r: -r["tag_score"])


# ==========================================================================
# LAB 4 — retrieval by similarity
# ==========================================================================

STOPWORDS = {"the", "a", "an", "and", "or", "of", "to", "in", "on", "at", "for", "was",
             "were", "is", "are", "be", "been", "by", "with", "that", "this", "it",
             "as", "from", "but", "not", "no", "we", "our", "its", "their", "has",
             "had", "have", "than", "then", "so", "which", "when", "all", "one"}


def tokenize(text: str) -> List[str]:
    return [t for t in re.findall(r"[a-z0-9\-]+", text.lower())
            if t not in STOPWORDS and len(t) > 2]


class SimilarityIndex:
    """TF-IDF vectors + cosine similarity over the archive's text."""

    def __init__(self, records: List[Dict[str, Any]]):
        self.records = records
        self.docs = [self._text(r) for r in records]
        self.tokens = [tokenize(d) for d in self.docs]
        n = len(self.tokens)
        df = Counter(t for toks in self.tokens for t in set(toks))
        # Smoothed IDF, so a term in every document still carries a little weight.
        self.idf = {t: math.log((n + 1) / (c + 1)) + 1.0 for t, c in df.items()}
        self.vectors = [self._vector(toks) for toks in self.tokens]

    @staticmethod
    def _text(rec: Dict[str, Any]) -> str:
        parts = [rec.get("title", ""), rec.get("summary", ""),
                 " ".join(rec.get("tags", [])), " ".join(rec.get("decisions", [])),
                 rec.get("outcome", "")]
        return " ".join(p for p in parts if p)

    def _vector(self, toks: List[str]) -> Dict[str, float]:
        if not toks:
            return {}
        tf = Counter(toks)
        vec = {t: (c / len(toks)) * self.idf.get(t, 1.0) for t, c in tf.items()}
        norm = math.sqrt(sum(v * v for v in vec.values())) or 1.0
        return {t: v / norm for t, v in vec.items()}

    @staticmethod
    def _cosine(a: Dict[str, float], b: Dict[str, float]) -> float:
        small, large = (a, b) if len(a) < len(b) else (b, a)
        return sum(v * large.get(t, 0.0) for t, v in small.items())

    def query(self, text: str, k: int = 3,
              record_type: Optional[str] = None) -> List[Dict[str, Any]]:
        qv = self._vector(tokenize(text))
        scored = []
        for rec, vec in zip(self.records, self.vectors):
            if record_type and rec.get("type") != record_type:
                continue
            scored.append({**rec, "similarity": round(self._cosine(qv, vec), 4)})
        return sorted(scored, key=lambda r: -r["similarity"])[:k]


index = SimilarityIndex(archive.all())


# ==========================================================================
# LAB 4 — hybrid retrieval
# ==========================================================================

def retrieve(store: Archive, idx: SimilarityIndex, text: str,
             tags: Optional[List[str]] = None, k: int = 3,
             record_type: Optional[str] = None) -> List[Dict[str, Any]]:
    """Tag filter first (precision), then similarity rank (relevance)."""
    pool = (retrieve_by_tags(store, tags, "any", record_type) if tags
            else [r for r in store.all()
                  if not record_type or r.get("type") == record_type])
    if not pool:
        return []
    sub = SimilarityIndex(pool)
    hits = sub.query(text, k=k, record_type=record_type)
    by_id = {h["record_id"]: h for h in hits}
    for p in pool:
        if p["record_id"] in by_id and "tag_score" in p:
            by_id[p["record_id"]]["tag_score"] = p["tag_score"]
    return hits


# ==========================================================================
# LAB 5 — the MCP protocol
# ==========================================================================

# ---- error codes: one vocabulary for every backend ------------------------
E_UNKNOWN_RESOURCE = "UNKNOWN_RESOURCE"
E_UNAUTHORIZED     = "UNAUTHORIZED"
E_FORBIDDEN_SCOPE  = "FORBIDDEN_SCOPE"
E_BAD_REQUEST      = "BAD_REQUEST"
E_NOT_FOUND        = "NOT_FOUND"
E_BACKEND_FAILURE  = "BACKEND_FAILURE"


class MCPError(Exception):
    def __init__(self, code: str, message: str):
        super().__init__(f"{code}: {message}")
        self.code, self.message = code, message


@dataclass
class Resource:
    """One advertised capability of the server."""
    name: str
    description: str
    params: Dict[str, Any]          # typed parameter schema
    returns: str
    scope: str                      # the scope a token must hold
    backend: str                    # which backend currently serves it
    handler: Callable = None

    def manifest(self) -> Dict[str, Any]:
        return {"name": self.name, "description": self.description,
                "params": self.params, "returns": self.returns,
                "required_scope": self.scope, "served_by": self.backend}


class MCPServer:
    """One server, many backends. Agents never see which is which."""

    def __init__(self, name: str):
        self.name = name
        self.resources: Dict[str, Resource] = {}
        self.tokens: Dict[str, Dict[str, Any]] = {}
        self.log: List[Dict[str, Any]] = []

    # ---- registration & auth -------------------------------------------
    def expose(self, resource: Resource) -> None:
        self.resources[resource.name] = resource

    def issue_token(self, token: str, holder: str, scopes: List[str]) -> None:
        self.tokens[token] = {"holder": holder, "scopes": set(scopes)}

    def describe(self) -> Dict[str, Any]:
        """Capability discovery — what a client may call, without calling it."""
        return {"server": self.name,
                "resources": [r.manifest() for r in self.resources.values()]}

    # ---- the single entry point ----------------------------------------
    def call(self, token: str, resource: str, **params) -> Dict[str, Any]:
        started = time.perf_counter()
        entry = {"seq": len(self.log) + 1, "resource": resource,
                 "holder": self.tokens.get(token, {}).get("holder", "<unknown>"),
                 "params": {k: (f"<{len(v)} rows>" if isinstance(v, list) else v)
                            for k, v in params.items()},
                 "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}

        def finish(ok, code=None, note=""):
            entry.update({"ok": ok, "error_code": code, "note": note,
                          "ms": round((time.perf_counter() - started) * 1000, 2)})
            self.log.append(entry)

        if token not in self.tokens:
            finish(False, E_UNAUTHORIZED)
            raise MCPError(E_UNAUTHORIZED, "token not recognised by this server")
        if resource not in self.resources:
            finish(False, E_UNKNOWN_RESOURCE)
            raise MCPError(E_UNKNOWN_RESOURCE,
                           f"'{resource}' is not exposed; call describe() to see "
                           f"{sorted(self.resources)}")
        res = self.resources[resource]
        if res.scope not in self.tokens[token]["scopes"]:
            finish(False, E_FORBIDDEN_SCOPE)
            raise MCPError(E_FORBIDDEN_SCOPE,
                           f"'{resource}' needs scope '{res.scope}'; this token holds "
                           f"{sorted(self.tokens[token]['scopes'])}")
        for p, spec in res.params.items():
            if spec.get("required") and p not in params:
                finish(False, E_BAD_REQUEST)
                raise MCPError(E_BAD_REQUEST, f"{resource}: missing parameter '{p}'")
        try:
            data = res.handler(**params)
        except MCPError as exc:
            finish(False, exc.code, exc.message)
            raise
        except Exception as exc:
            finish(False, E_BACKEND_FAILURE, str(exc))
            raise MCPError(E_BACKEND_FAILURE, f"{res.backend} failed: {exc}")
        entry["served_by"] = res.backend
        finish(True)
        # The envelope is identical whatever the backend was.
        return {"ok": True, "resource": resource, "served_by": res.backend, "data": data,
                "meta": {"server": self.name, "at": entry["at"]}}


class MCPClient:
    """What an agent holds. It knows a token and resource names — nothing else."""

    def __init__(self, server: MCPServer, token: str):
        self._server, self._token = server, token

    def describe(self) -> Dict[str, Any]:
        return self._server.describe()

    def read(self, resource: str, **params) -> Any:
        return self._server.call(self._token, resource, **params)["data"]


# ==========================================================================
# LAB 5 — file, SQLite and API backends
# ==========================================================================

WORKDIR = Path("lab5_sources")
WORKDIR.mkdir(exist_ok=True)

# ---- backend 1: a FILE ----------------------------------------------------
CSV_PATH = WORKDIR / "department_reports_2025_26.csv"
CSV_PATH.write_text(DEPARTMENT_REPORTS_CSV)


def _cast(row: Dict[str, str]) -> Dict[str, Any]:
    out = dict(row)
    for col, caster in NUMERIC_COLUMNS.items():
        if col in out:
            out[col] = caster(out[col])
    return out


class FileBackend:
    name = "file:csv"

    def __init__(self, path: Path):
        self.path = path

    def department_reports(self, period: str, category: str = None) -> Dict[str, Any]:
        if period != REPORTING_PERIOD:
            raise MCPError(E_NOT_FOUND, f"no export on disk for '{period}'")
        with self.path.open() as fh:
            rows = [_cast(r) for r in csv.DictReader(fh)]
        if category:
            rows = [r for r in rows if r["category"] == category]
            if not rows:
                raise MCPError(E_NOT_FOUND, f"no departments in category '{category}'")
        return {"rows": rows, "row_count": len(rows), "period": period,
                "source_system": "Department reports export (CSV on disk)"}


# ---- backend 2: SQLite ----------------------------------------------------
DB_PATH = WORKDIR / "institution.db"
if DB_PATH.exists():
    DB_PATH.unlink()
conn = sqlite3.connect(DB_PATH)
conn.executescript("""
CREATE TABLE kpi_definition (
    kpi_id TEXT PRIMARY KEY, name TEXT, domain TEXT, unit TEXT, formula TEXT,
    target REAL, target_source TEXT, source_system TEXT, figure_status TEXT, caveat TEXT);
CREATE TABLE department_report (
    dept_code TEXT PRIMARY KEY, dept_name TEXT, category TEXT, period TEXT,
    payload TEXT);
""")
conn.executemany(
    "INSERT INTO kpi_definition VALUES (?,?,?,?,?,?,?,?,?,?)",
    [(k, d["name"], d["domain"], d["unit"], d["formula"], d["target"],
      d["target_source"], d["source_system"], d.get("figure_status", "final"),
      d.get("caveat")) for k, d in KPI_DEFINITIONS.items()])
conn.executemany(
    "INSERT INTO department_report VALUES (?,?,?,?,?)",
    [(r["dept_code"], r["dept_name"], r["category"], REPORTING_PERIOD, json.dumps(r))
     for r in (_cast(x) for x in csv.DictReader(io.StringIO(DEPARTMENT_REPORTS_CSV)))])
conn.commit()


class SQLiteBackend:
    name = "sqlite"

    def __init__(self, path: Path):
        self.path = path

    def _rows(self, sql: str, args=()) -> List[sqlite3.Row]:
        c = sqlite3.connect(self.path)
        c.row_factory = sqlite3.Row
        try:
            return c.execute(sql, args).fetchall()
        finally:
            c.close()

    def kpi_definitions(self, kpi_ids: List[str] = None) -> Dict[str, Any]:
        if kpi_ids:
            marks = ",".join("?" * len(kpi_ids))
            rows = self._rows(
                f"SELECT * FROM kpi_definition WHERE kpi_id IN ({marks})", kpi_ids)
            if len(rows) != len(kpi_ids):
                found = {r["kpi_id"] for r in rows}
                raise MCPError(E_NOT_FOUND,
                               f"unknown KPI id(s): {sorted(set(kpi_ids) - found)}")
        else:
            rows = self._rows("SELECT * FROM kpi_definition ORDER BY kpi_id")
        defs = {}
        for r in rows:
            d = dict(KPI_DEFINITIONS[r["kpi_id"]])   # formula fields for the calculator
            d.update({"name": r["name"], "domain": r["domain"], "target": r["target"],
                      "target_source": r["target_source"],
                      "source_system": r["source_system"]})
            defs[r["kpi_id"]] = d
        return {"definitions": defs, "count": len(defs),
                "source_system": "Institutional KPI register (SQLite)"}

    # The SAME resource the file backend serves — used in Cell 6 to swap sources.
    def department_reports(self, period: str, category: str = None) -> Dict[str, Any]:
        sql = "SELECT payload FROM department_report WHERE period = ?"
        args = [period]
        if category:
            sql += " AND category = ?"
            args.append(category)
        rows = [json.loads(r["payload"]) for r in self._rows(sql, args)]
        if not rows:
            raise MCPError(E_NOT_FOUND,
                           f"no rows for period '{period}'"
                           + (f" and category '{category}'" if category else ""))
        return {"rows": rows, "row_count": len(rows), "period": period,
                "source_system": "Department reports (SQLite table)"}


# ---- backend 3: a mock HTTP API -------------------------------------------
class APIBackend:
    name = "http:benchmark-api"
    LATENCY_S = 0.02

    def external_benchmark(self, kpi_id: str) -> Dict[str, Any]:
        time.sleep(self.LATENCY_S)              # a network call is not free
        if kpi_id not in KPI_DEFINITIONS:
            raise MCPError(E_BAD_REQUEST, f"'{kpi_id}' is not a known KPI id")
        record = BENCHMARK_API_DB.get(kpi_id)
        if record is None:
            raise MCPError(E_NOT_FOUND,
                           f"no published external benchmark for {kpi_id} "
                           f"({KPI_DEFINITIONS[kpi_id]['name']})")
        return {"kpi_id": kpi_id, **record,
                "source_system": "External benchmark API (NIRF cohort)"}


file_backend, sqlite_backend, api_backend = (FileBackend(CSV_PATH),
                                             SQLiteBackend(DB_PATH), APIBackend())


# ==========================================================================
# LAB 5 — expose resources, issue scoped tokens
# ==========================================================================

server = MCPServer("edss-connector")

server.expose(Resource(
    name="department_reports",
    description="Per-department metric rows for one reporting period.",
    params={"period": {"type": "string", "required": True},
            "category": {"type": "string", "required": False,
                         "enum": ["circuit", "core"]}},
    returns="{rows, row_count, period, source_system}",
    scope="read:departments", backend=file_backend.name,
    handler=file_backend.department_reports))

server.expose(Resource(
    name="kpi_definitions",
    description="Approved institution KPI definitions: formula, unit, target, authority.",
    params={"kpi_ids": {"type": "array", "required": False}},
    returns="{definitions, count, source_system}",
    scope="read:kpi-register", backend=sqlite_backend.name,
    handler=sqlite_backend.kpi_definitions))

server.expose(Resource(
    name="external_benchmark",
    description="Published peer median for a KPI. Coverage is partial.",
    params={"kpi_id": {"type": "string", "required": True}},
    returns="{kpi_id, peer_median, cohort, year, source_system}",
    scope="read:benchmarks", backend=api_backend.name,
    handler=api_backend.external_benchmark))

# ---- scoped tokens: the Registrar's authorisation, expressed as data -------
server.issue_token("tok-coordinator", "apniLeap AURA Coordinator",
                   ["read:departments", "read:kpi-register", "read:benchmarks"])
server.issue_token("tok-academic-agent", "Academic Agent",
                   ["read:departments", "read:kpi-register"])   # no benchmark access
server.issue_token("tok-intern", "Summer intern (read-only KPI register)",
                   ["read:kpi-register"])

# ==========================================================================
# A ready-made client for the coordinator
# ==========================================================================

coordinator_client = MCPClient(server, "tok-coordinator")


Writing edss_stack.py


### Cell 2 · What a runtime owns

In [3]:
# =============================================================================
# Cell 2  ·  What a runtime owns — budgets, audit, checkpoints
# =============================================================================
# Concept 6: a runtime is the controller the agent loop runs INSIDE. It binds
# skills (Lab 3), memory (Lab 4) and the connector (Lab 5) behind one entry
# point, and it owns four things the agent is not allowed to own itself:
#
#   budgets      — a run cannot spend more than it was granted
#   audit log    — every action, with its inputs and outputs
#   checkpoints  — state persisted after each step, so a crash resumes
#   gates        — actions that stop and wait for a named human

import json, textwrap, time, traceback
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

from edss_core import DOMAINS, KPI_DEFINITIONS, REPORTING_PERIOD, ToolRouter
from edss_stack import (SKILL_REGISTRY, RunState, Archive, archive, index,
                        retrieve_by_tags, server, coordinator_client, MCPError)


class BudgetExceeded(Exception):
    """Raised when a run tries to spend past its granted budget."""


class ApprovalRequired(Exception):
    """Raised when an action needs a named human and has not got one."""


@dataclass
class Budget:
    """What this run is allowed to spend. Checked before every action."""
    max_actions: int = 50
    max_connector_calls: int = 60
    max_seconds: float = 120.0

    def report(self, used_actions, used_calls, elapsed) -> str:
        return (f"actions {used_actions}/{self.max_actions} · "
                f"connector {used_calls}/{self.max_connector_calls} · "
                f"{elapsed:.2f}s/{self.max_seconds:.0f}s")


@dataclass
class Action:
    """One unit of work the runtime will perform, log and checkpoint."""
    name: str
    kind: str                       # tool | skill | memory | compute | gate | publish
    description: str
    fn: Callable
    requires_approval: bool = False
    depends_on: List[str] = field(default_factory=list)


class CheckpointStore:
    """Run state persisted to disk after every completed action."""

    def __init__(self, directory: Path):
        self.dir = directory
        self.dir.mkdir(exist_ok=True)

    def path(self, run_id: str) -> Path:
        return self.dir / f"{run_id}.checkpoint.json"

    def save(self, run_id: str, payload: Dict[str, Any]) -> Path:
        p = self.path(run_id)
        p.write_text(json.dumps(payload, indent=2, default=str))
        return p

    def load(self, run_id: str) -> Optional[Dict[str, Any]]:
        p = self.path(run_id)
        return json.loads(p.read_text()) if p.exists() else None

    def exists(self, run_id: str) -> bool:
        return self.path(run_id).exists()


print("Runtime primitives declared:")
print("  Budget          — actions, connector calls and wall time, checked per action")
print("  Action          — a named unit of work with its dependencies and gate flag")
print("  CheckpointStore — run state written to disk after every completed action")


Runtime primitives declared:
  Budget          — actions, connector calls and wall time, checked per action
  Action          — a named unit of work with its dependencies and gate flag
  CheckpointStore — run state written to disk after every completed action


### Cell 3 · The runtime

Read `perform()`. In order, it: skips work a checkpoint says is done, refuses an action whose dependencies have not run, refuses it if the budget is spent, refuses it if it needs a human and has none, logs inputs and outputs either way, and checkpoints to disk — **including on failure**.

That last detail is what makes Cell 6 possible.

In [4]:
# =============================================================================
# Cell 3  ·  The runtime
# =============================================================================
# One entry point: perform(). Nothing in this worklet does work outside it, so
# the audit log is complete by construction rather than by discipline.

class Runtime:
    """OpenCLA/OpenClaw-style runtime: skills + tools + memory + audit + gates."""

    def __init__(self, run_id: str, budget: Budget, checkpoints: CheckpointStore,
                 client=coordinator_client, skills=SKILL_REGISTRY,
                 store: Archive = archive):
        self.run_id = run_id
        self.budget = budget
        self.checkpoints = checkpoints
        self.client = client                 # Lab 5 — the only door to data
        self.skills = skills                 # Lab 3
        self.store = store                   # Lab 4
        self.router = ToolRouter(agent=f"runtime:{run_id}")
        self.state = RunState(run_id=run_id, period=REPORTING_PERIOD)
        self.audit: List[Dict[str, Any]] = []
        self.completed: List[str] = []
        self.results: Dict[str, Any] = {}
        self.approvals: Dict[str, str] = {}
        self.started = time.perf_counter()
        self.connector_calls_at_start = len(server.log)

    # ---- accounting ----------------------------------------------------
    @property
    def elapsed(self) -> float:
        return time.perf_counter() - self.started

    @property
    def connector_calls(self) -> int:
        return len(server.log) - self.connector_calls_at_start

    def _check_budget(self, action_name: str) -> None:
        if len(self.completed) >= self.budget.max_actions:
            raise BudgetExceeded(f"{action_name}: action budget spent "
                                 f"({self.budget.max_actions})")
        if self.connector_calls >= self.budget.max_connector_calls:
            raise BudgetExceeded(f"{action_name}: connector-call budget spent "
                                 f"({self.budget.max_connector_calls})")
        if self.elapsed >= self.budget.max_seconds:
            raise BudgetExceeded(f"{action_name}: wall-time budget spent "
                                 f"({self.budget.max_seconds}s)")

    # ---- audit ---------------------------------------------------------
    def _log(self, action: Action, status: str, inputs, output, error=None,
             ms: float = 0.0) -> None:
        self.audit.append({
            "seq": len(self.audit) + 1, "run_id": self.run_id, "action": action.name,
            "kind": action.kind, "status": status,
            "inputs": inputs, "output": output, "error": error,
            "ms": round(ms, 2),
            "budget": self.budget.report(len(self.completed), self.connector_calls,
                                         self.elapsed),
            "at": datetime.now(timezone.utc).isoformat(timespec="seconds")})

    @staticmethod
    def _summarise(value, depth=0):
        if isinstance(value, dict):
            return {k: Runtime._summarise(v, depth + 1) for k, v in list(value.items())[:8]}
        if isinstance(value, list):
            return f"<{len(value)} items>" if len(value) > 4 else [
                Runtime._summarise(v, depth + 1) for v in value]
        if isinstance(value, str) and len(value) > 120:
            return value[:117] + "..."
        return value

    # ---- the single entry point ----------------------------------------
    def perform(self, action: Action, **inputs) -> Any:
        if action.name in self.completed:               # resume: never redo work
            self._log(action, "skipped (already completed)", inputs,
                      self._summarise(self.results.get(action.name)))
            return self.results[action.name]

        missing = [d for d in action.depends_on if d not in self.completed]
        if missing:
            raise RuntimeError(f"{action.name}: depends on {missing}, not yet completed")

        self._check_budget(action.name)

        if action.requires_approval and action.name not in self.approvals:
            self._log(action, "blocked: awaiting approval", inputs, None)
            raise ApprovalRequired(
                f"{action.name} requires a named human. Call runtime.approve("
                f"'{action.name}', by='...') and run again.")

        t0 = time.perf_counter()
        try:
            output = action.fn(self, **inputs)
        except Exception as exc:
            self._log(action, "failed", inputs, None, error=f"{type(exc).__name__}: {exc}",
                      ms=(time.perf_counter() - t0) * 1000)
            self.checkpoint()                            # a crash still checkpoints
            raise
        self.results[action.name] = output
        self.completed.append(action.name)
        self._log(action, "ok", inputs, self._summarise(output),
                  ms=(time.perf_counter() - t0) * 1000)
        self.checkpoint()
        return output

    # ---- gates ---------------------------------------------------------
    def approve(self, action_name: str, by: str) -> None:
        self.approvals[action_name] = by
        self.audit.append({"seq": len(self.audit) + 1, "run_id": self.run_id,
                           "action": action_name, "kind": "gate", "status": "approved",
                           "inputs": {"by": by}, "output": None, "error": None, "ms": 0.0,
                           "budget": "-",
                           "at": datetime.now(timezone.utc).isoformat(timespec="seconds")})

    # ---- checkpoint / resume -------------------------------------------
    def checkpoint(self) -> Path:
        return self.checkpoints.save(self.run_id, {
            "run_id": self.run_id, "period": self.state.period,
            "completed": self.completed, "results": self.results,
            "approvals": self.approvals, "audit": self.audit,
            "saved_at": datetime.now(timezone.utc).isoformat(timespec="seconds")})

    @classmethod
    def resume(cls, run_id: str, budget: Budget, checkpoints: CheckpointStore) -> "Runtime":
        snap = checkpoints.load(run_id)
        if snap is None:
            raise FileNotFoundError(f"no checkpoint for run '{run_id}'")
        rt = cls(run_id, budget, checkpoints)
        rt.completed = list(snap["completed"])
        rt.results = dict(snap["results"])
        rt.approvals = dict(snap["approvals"])
        rt.audit = list(snap["audit"])
        rt.audit.append({"seq": len(rt.audit) + 1, "run_id": run_id, "action": "-",
                         "kind": "runtime", "status": "resumed from checkpoint",
                         "inputs": {"completed_before_crash": rt.completed},
                         "output": None, "error": None, "ms": 0.0, "budget": "-",
                         "at": datetime.now(timezone.utc).isoformat(timespec="seconds")})
        return rt


print("Runtime ready. Every action passes through perform(), which:")
print("  1. skips it if a checkpoint says it is already done  (resume)")
print("  2. refuses it if a dependency has not run            (ordering)")
print("  3. refuses it if the budget is spent                 (budgets)")
print("  4. refuses it if it needs a human and has none       (gates)")
print("  5. logs inputs and outputs either way                (audit)")
print("  6. checkpoints to disk — including on failure        (durability)")


Runtime ready. Every action passes through perform(), which:
  1. skips it if a checkpoint says it is already done  (resume)
  2. refuses it if a dependency has not run            (ordering)
  3. refuses it if the budget is spent                 (budgets)
  4. refuses it if it needs a human and has none       (gates)
  5. logs inputs and outputs either way                (audit)
  6. checkpoints to disk — including on failure        (durability)


### Cell 4 · The run plan

Seven actions spanning all three earlier labs: the connector for data (Lab 5), the skill registry for work (Lab 3), the archive for baseline (Lab 4).

`FAULT` arms a single transient backend failure so the crash path is real rather than described.

In [5]:
# =============================================================================
# Cell 4  ·  The run plan — skills, tools and memory, all bound by the runtime
# =============================================================================
# Seven actions. Each one reaches its capability through the runtime: the
# connector for data (Lab 5), the skill registry for work (Lab 3), the archive
# for baseline (Lab 4). None of them touches a source or a store directly.

# A transient backend failure, armed once, to prove the crash/resume path.
FAULT = {"armed": True}


def a_load_departments(rt, **_):
    reports = rt.client.read("department_reports", period=rt.state.period)
    rt.state.note("row_count", reports["row_count"])
    return {"row_count": reports["row_count"], "source": reports["source_system"]}


def a_review_domains(rt, **_):
    out = {}
    for domain in DOMAINS:
        review = rt.skills["review_domain"].run(rt.router, domain=domain,
                                                period=rt.state.period)
        out[domain] = {"headline": review["headline"],
                       "off_target": review["off_target"],
                       "kpis": {k["kpi_id"]: k["value"] for k in review["kpis"]}}
    return out


def a_retrieve_baseline(rt, **_):
    snaps = retrieve_by_tags(rt.store, ["kpi-snapshot"], "any")
    return {s["period"]: s["values"] for s in snaps}


def a_compute_trend(rt, **_):
    if FAULT["armed"]:                      # <-- the simulated crash
        FAULT["armed"] = False
        raise MCPError("BACKEND_FAILURE",
                       "benchmark service dropped the connection mid-run")
    current = {k: v for d in rt.results["review_domains"].values()
               for k, v in d["kpis"].items()}
    base = rt.results["retrieve_baseline"]["2024-25"]
    return {k: {"now": round(current[k], 2), "prior": base[k],
                "yoy": round(current[k] - base[k], 2),
                "direction": "improving" if current[k] > base[k] else "declining"}
            for k in current}


def a_assemble_brief(rt, **_):
    trend = rt.results["compute_trend"]
    declining = [k for k, t in trend.items() if t["direction"] == "declining"]
    off = sorted({k for d in rt.results["review_domains"].values()
                  for k in d["off_target"]})
    doc = rt.skills["format_document"].run(
        rt.router,
        title=f"Executive decision brief — {rt.state.period}",
        meta={"Run": rt.run_id, "Status": "DRAFT — not circulated"},
        template="brief",
        sections=[
            {"heading": "Headline", "bullets":
                [d["headline"] for d in rt.results["review_domains"].values()]},
            {"heading": "Off target", "bullets": off},
            {"heading": "Declining year on year", "bullets": declining or ["none"]}])
    rt.state.draft = {"document": doc["document"], "status": "DRAFT"}
    return {"lines": doc["line_count"], "off_target": off, "declining": declining}


def a_human_gate(rt, **_):
    who = rt.approvals["human_gate"]
    rt.state.draft["status"] = "APPROVED"
    return {"approved_by": who, "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}


def a_publish(rt, **_):
    return {"published": True, "version": 1,
            "circulated_to": "Governing Council",
            "approved_by": rt.results["human_gate"]["approved_by"]}


PLAN = [
    Action("load_departments", "tool", "Read department rows through the connector",
           a_load_departments),
    Action("review_domains", "skill", "Run the review_domain skill for all four domains",
           a_review_domains, depends_on=["load_departments"]),
    Action("retrieve_baseline", "memory", "Read closed-year KPI snapshots from the archive",
           a_retrieve_baseline, depends_on=["load_departments"]),
    Action("compute_trend", "compute", "Year-on-year movement per KPI",
           a_compute_trend, depends_on=["review_domains", "retrieve_baseline"]),
    Action("assemble_brief", "skill", "Render the brief with the format_document skill",
           a_assemble_brief, depends_on=["compute_trend"]),
    Action("human_gate", "gate", "Principal / Registrar approval",
           a_human_gate, requires_approval=True, depends_on=["assemble_brief"]),
    Action("publish", "publish", "Release the approved, versioned brief",
           a_publish, depends_on=["human_gate"]),
]

print(f"{'action':<20}{'kind':<10}{'gate':<7}depends on")
for a in PLAN:
    print(f"{a.name:<20}{a.kind:<10}{'yes' if a.requires_approval else '-':<7}"
          f"{a.depends_on or '-'}")


action              kind      gate   depends on
load_departments    tool      -      -
review_domains      skill     -      ['load_departments']
retrieve_baseline   memory    -      ['load_departments']
compute_trend       compute   -      ['review_domains', 'retrieve_baseline']
assemble_brief      skill     -      ['compute_trend']
human_gate          gate      yes    ['assemble_brief']
publish             publish   -      ['human_gate']


### Cell 5 · Run 1 — and it crashes

Three actions succeed, then the benchmark service drops the connection. Note that the last audit entry **is the failure**, with its error text — the crash is part of the record, not a gap in it.

In [6]:
# =============================================================================
# Cell 5  ·  Run 1 — and it crashes
# =============================================================================
# The backend drops mid-run, at compute_trend. Three actions are already done.
# A runtime without checkpoints loses all three.

checkpoints = CheckpointStore(Path("lab6_checkpoints"))
budget = Budget(max_actions=50, max_connector_calls=60, max_seconds=120)

RUN_ID = "EDSS-RUNTIME-DEMO"
if checkpoints.exists(RUN_ID):
    checkpoints.path(RUN_ID).unlink()

rt = Runtime(RUN_ID, budget, checkpoints)

print("=" * 86)
print("RUN 1")
print("=" * 86)
crashed_at = None
for action in PLAN:
    try:
        rt.perform(action)
        print(f"  [ok]      {action.name:<20} {rt.budget.report(len(rt.completed), rt.connector_calls, rt.elapsed)}")
    except ApprovalRequired as exc:
        print(f"  [gate]    {action.name:<20} {exc}")
        break
    except Exception as exc:
        crashed_at = action.name
        print(f"  [CRASH]   {action.name:<20} {type(exc).__name__}: {exc}")
        break

print(f"\n  completed before the crash : {rt.completed}")
print(f"  checkpoint on disk         : {checkpoints.path(RUN_ID).name} "
      f"({checkpoints.path(RUN_ID).stat().st_size:,} bytes)")
print(f"  audit entries              : {len(rt.audit)}")
print(f"\n  Note the last audit entry is the FAILURE, with its error recorded —")
print(f"  the crash is part of the record, not a gap in it:")
last = rt.audit[-1]
print(f"      action={last['action']}  status={last['status']}")
print(f"      error ={last['error']}")


RUN 1
  [ok]      load_departments     actions 1/50 · connector 1/60 · 0.00s/120s
  [ok]      review_domains       actions 2/50 · connector 1/60 · 0.00s/120s
  [ok]      retrieve_baseline    actions 3/50 · connector 1/60 · 0.00s/120s
  [CRASH]   compute_trend        MCPError: BACKEND_FAILURE: benchmark service dropped the connection mid-run

  completed before the crash : ['load_departments', 'review_domains', 'retrieve_baseline']
  checkpoint on disk         : EDSS-RUNTIME-DEMO.checkpoint.json (5,535 bytes)
  audit entries              : 4

  Note the last audit entry is the FAILURE, with its error recorded —
  the crash is part of the record, not a gap in it:
      action=compute_trend  status=failed
      error =MCPError: BACKEND_FAILURE: benchmark service dropped the connection mid-run


### Cell 6 · Resume — the acceptance criterion

A **new** `Runtime` object is built from the checkpoint file, as it would be after a process death. The three completed actions are skipped; `compute_trend` is retried and succeeds; the run stops at the human gate.

Then the gate is approved by name and the run finishes. `re-executed despite already being done` must be empty — that is the check.

In [7]:
# =============================================================================
# Cell 6  ·  Resume — the acceptance criterion
# =============================================================================
# "A crashed run resumes from its last checkpoint." A brand-new Runtime object
# is built from the file on disk — the first one is gone, as it would be after
# a process death — and the three completed actions are NOT redone.

rt2 = Runtime.resume(RUN_ID, budget, checkpoints)

print("=" * 86)
print("RUN 2 — resumed from the checkpoint")
print("=" * 86)
print(f"  restored from disk : {rt2.completed}")
print(f"  audit carried over : {len(rt2.audit)} entries\n")

skipped, freshly_done = [], []
for action in PLAN:
    try:
        before = len(rt2.completed)
        rt2.perform(action)
        if len(rt2.completed) == before:
            skipped.append(action.name)
            print(f"  [skip]    {action.name:<20} already completed before the crash")
        else:
            freshly_done.append(action.name)
            print(f"  [ok]      {action.name:<20} "
                  f"{rt2.budget.report(len(rt2.completed), rt2.connector_calls, rt2.elapsed)}")
    except ApprovalRequired as exc:
        print(f"  [GATE]    {action.name:<20} stopped — {str(exc).splitlines()[0]}")
        break

print(f"\n  skipped because the checkpoint says they are done : {skipped}")
print(f"  re-executed despite already being done            : "
      f"{[a for a in freshly_done if a in rt.completed]}  <- must be empty")
print(f"  newly executed after the resume                   : {freshly_done}")

# ---- the human gate, then the rest of the run -----------------------------
print("\n" + "=" * 86)
print("THE HUMAN GATE")
print("=" * 86)
print("  The runtime stopped at human_gate and will not pass it on its own.")
rt2.approve("human_gate", by="Dr. A. Principal")
print("  runtime.approve('human_gate', by='Dr. A. Principal')\n")

for action in PLAN:
    if action.name in rt2.completed:
        continue
    rt2.perform(action)
    print(f"  [ok]      {action.name:<20} {rt2.results[action.name]}")

print(f"\n  run complete: {len(rt2.completed)}/{len(PLAN)} actions")
print(f"  brief status: {rt2.state.draft['status'] if rt2.state.draft else '—'}")


RUN 2 — resumed from the checkpoint
  restored from disk : ['load_departments', 'review_domains', 'retrieve_baseline']
  audit carried over : 5 entries

  [skip]    load_departments     already completed before the crash
  [skip]    review_domains       already completed before the crash
  [skip]    retrieve_baseline    already completed before the crash
  [ok]      compute_trend        actions 4/50 · connector 0/60 · 0.00s/120s
  [ok]      assemble_brief       actions 5/50 · connector 0/60 · 0.00s/120s
  [GATE]    human_gate           stopped — human_gate requires a named human. Call runtime.approve('human_gate', by='...') and run again.

  skipped because the checkpoint says they are done : ['load_departments', 'review_domains', 'retrieve_baseline']
  re-executed despite already being done            : []  <- must be empty
  newly executed after the resume                   : ['compute_trend', 'assemble_brief']

THE HUMAN GATE
  The runtime stopped at human_gate and will not pass it 

### Cell 7 · The runtime says no

A runtime that only logs is a logger. Four refusals make it a controller: a 2-action budget, a zero connector-call budget, an action whose dependencies have not run, and a human gate with no named human.

In [8]:
# =============================================================================
# Cell 7  ·  Budgets and ordering — the runtime says no
# =============================================================================
# A runtime that only logs is a logger. These are the refusals that make it a
# controller.

print("=" * 86)
print("BUDGET ENFORCEMENT")
print("=" * 86)
tiny = Runtime("EDSS-TINY-BUDGET", Budget(max_actions=2, max_connector_calls=60,
                                          max_seconds=120), checkpoints)
for action in PLAN:
    try:
        tiny.perform(action)
        print(f"  [ok]      {action.name}")
    except BudgetExceeded as exc:
        print(f"  [REFUSED] {exc}")
        break
    except Exception as exc:
        print(f"  [stopped] {action.name}: {type(exc).__name__}")
        break

print("\n" + "=" * 86)
print("CONNECTOR BUDGET")
print("=" * 86)
starved = Runtime("EDSS-NO-CONNECTOR", Budget(max_actions=50, max_connector_calls=0,
                                              max_seconds=120), checkpoints)
try:
    starved.perform(PLAN[0])
    print("  [!!] a run with a zero connector budget still read data")
except BudgetExceeded as exc:
    print(f"  [REFUSED] {exc}")

print("\n" + "=" * 86)
print("DEPENDENCY ORDERING")
print("=" * 86)
loose = Runtime("EDSS-OUT-OF-ORDER", budget, checkpoints)
ordering_refusal = None
try:
    loose.perform(PLAN[4])          # assemble_brief, with nothing before it
    print("  [!!] assemble_brief ran with no data behind it")
except RuntimeError as exc:
    ordering_refusal = str(exc)
    print(f"  [REFUSED] {exc}")

print("\n" + "=" * 86)
print("THE GATE CANNOT BE WALKED PAST")
print("=" * 86)
gated = Runtime("EDSS-NO-APPROVAL", budget, checkpoints)
for action in PLAN[:5]:
    try:
        gated.perform(action)
    except Exception:
        break
try:
    gated.perform(PLAN[5])
    print("  [!!] the human gate was passed without a named human")
except ApprovalRequired as exc:
    print(f"  [REFUSED] {str(exc).splitlines()[0]}")
except Exception as exc:
    print(f"  [stopped earlier] {type(exc).__name__}: {exc}")

print("\n" + "=" * 86)
print(f"AUDIT LOG — RUN {RUN_ID} ({len(rt2.audit)} entries)")
print("=" * 86)
print(f"  {'#':<4}{'action':<20}{'kind':<10}{'status':<32}{'ms':>8}")
for e in rt2.audit:
    print(f"  {e['seq']:<4}{e['action'][:18]:<20}{e['kind']:<10}{e['status'][:30]:<32}"
          f"{e['ms']:>8.2f}")


BUDGET ENFORCEMENT
  [ok]      load_departments
  [ok]      review_domains
  [REFUSED] retrieve_baseline: action budget spent (2)

CONNECTOR BUDGET
  [REFUSED] load_departments: connector-call budget spent (0)

DEPENDENCY ORDERING
  [REFUSED] assemble_brief: depends on ['compute_trend'], not yet completed

THE GATE CANNOT BE WALKED PAST
  [REFUSED] human_gate requires a named human. Call runtime.approve('human_gate', by='...') and run again.

AUDIT LOG — RUN EDSS-RUNTIME-DEMO (14 entries)
  #   action              kind      status                                ms
  1   load_departments    tool      ok                                  2.55
  2   review_domains      skill     ok                                  0.91
  3   retrieve_baseline   memory    ok                                  0.10
  4   compute_trend       compute   failed                              0.01
  5   -                   runtime   resumed from checkpoint             0.00
  6   load_departments    tool      skipped 

### Cell 8 · Acceptance checks — the rubric, executed

In [9]:
# =============================================================================
# Cell 8  ·  Acceptance checks — the rubric, executed
# =============================================================================
# Worklet 15, Lab 6 acceptance: "Every action is logged with inputs/outputs;
# a crashed run resumes from its last checkpoint."

checks = []


def check(name, condition, evidence=""):
    checks.append((name, bool(condition), evidence))


work = [e for e in rt2.audit if e["kind"] != "runtime"]
check("F1  every action is logged with its inputs AND outputs",
      work and all({"action", "kind", "status", "inputs", "output", "ms"} <= set(e)
                   for e in work),
      f"{len(work)} action records, each carrying inputs, output, status and duration")

check("F2  the failure is in the log, not a gap in it",
      any(e["status"] == "failed" and e["error"] for e in rt2.audit),
      next(e["error"] for e in rt2.audit if e["status"] == "failed"))

check("F3  the crash wrote a checkpoint",
      checkpoints.exists(RUN_ID),
      f"{checkpoints.path(RUN_ID).name}, "
      f"{checkpoints.path(RUN_ID).stat().st_size:,} bytes on disk")

check("F4  a NEW runtime object was rebuilt from that file",
      rt2 is not rt and rt2.completed[:3] == rt.completed[:3],
      f"resumed with {rt.completed} already done")

check("F5  completed work was not repeated after the resume",
      not [a for a in freshly_done if a in rt.completed],
      f"skipped {len(skipped)} completed actions; re-executed 0 of them")

check("F6  the resumed run finished every action",
      len(rt2.completed) == len(PLAN),
      f"{len(rt2.completed)}/{len(PLAN)} actions complete")

check("F7  an action cannot run before its dependencies",
      ordering_refusal is not None and "depends on" in ordering_refusal
      and "assemble_brief" not in loose.completed,
      ordering_refusal or "NO REFUSAL RAISED")

check("F8  the action budget is enforced",
      len(tiny.completed) <= 2,
      f"a 2-action budget stopped the run after {len(tiny.completed)} actions")

check("F9  the connector budget is enforced independently",
      len(starved.completed) == 0,
      "a zero connector-call budget refused the very first read")

check("F10 the human gate cannot be passed without a named human",
      "human_gate" not in gated.completed,
      "the ungated runtime never completed human_gate")

check("F11 the gate, once approved, records WHO approved it",
      rt2.results["human_gate"]["approved_by"] == "Dr. A. Principal"
      and any(e["status"] == "approved" for e in rt2.audit),
      f"approved_by = {rt2.results['human_gate']['approved_by']}, recorded in the audit log")

check("F12 nothing is published before the gate",
      rt2.completed.index("human_gate") < rt2.completed.index("publish"),
      f"order: …{rt2.completed[-3:]}")

check("F13 the runtime binds all three earlier labs",
      any(e["kind"] == "skill" for e in rt2.audit)
      and any(e["kind"] == "memory" for e in rt2.audit)
      and any(e["kind"] == "tool" for e in rt2.audit),
      "skills (Lab 3), archive (Lab 4) and connector (Lab 5) all driven through perform()")

print("=" * 86)
print("LAB 6 ACCEPTANCE CHECKS")
print("=" * 86)
for name, ok, ev in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}]  {name}")
    if ev:
        print(f"          {ev}")
failed = [n for n, ok, _ in checks if not ok]
print("=" * 86)
print(f"  {len(checks) - len(failed)}/{len(checks)} checks passed.")
assert not failed, f"FAILED: {failed}"
print("  Lab 6 acceptance criteria satisfied.")


LAB 6 ACCEPTANCE CHECKS
  [PASS]  F1  every action is logged with its inputs AND outputs
          13 action records, each carrying inputs, output, status and duration
  [PASS]  F2  the failure is in the log, not a gap in it
          MCPError: BACKEND_FAILURE: benchmark service dropped the connection mid-run
  [PASS]  F3  the crash wrote a checkpoint
          EDSS-RUNTIME-DEMO.checkpoint.json, 14,390 bytes on disk
  [PASS]  F4  a NEW runtime object was rebuilt from that file
          resumed with ['load_departments', 'review_domains', 'retrieve_baseline'] already done
  [PASS]  F5  completed work was not repeated after the resume
          skipped 3 completed actions; re-executed 0 of them
  [PASS]  F6  the resumed run finished every action
          7/7 actions complete
  [PASS]  F7  an action cannot run before its dependencies
          assemble_brief: depends on ['compute_trend'], not yet completed
  [PASS]  F8  the action budget is enforced
          a 2-action budget stopped th

### Cell 9 · Deliverables

In [10]:
# =============================================================================
# Cell 9  ·  Deliverables
# =============================================================================
outdir = Path("lab6_artifacts")
outdir.mkdir(exist_ok=True)

artifacts = {
    "lab6_run_plan.json": [{"name": a.name, "kind": a.kind,
                            "description": a.description,
                            "requires_approval": a.requires_approval,
                            "depends_on": a.depends_on} for a in PLAN],
    "lab6_audit_log.json": rt2.audit,
    "lab6_checkpoint.json": checkpoints.load(RUN_ID),
    "lab6_crash_and_resume.json": {
        "completed_before_crash": rt.completed,
        "crashed_at": crashed_at,
        "error": next(e["error"] for e in rt.audit if e["status"] == "failed"),
        "skipped_on_resume": skipped,
        "executed_after_resume": freshly_done,
        "final_completed": rt2.completed},
    "lab6_budget_refusals.json": {
        "action_budget": {"limit": 2, "completed": tiny.completed},
        "connector_budget": {"limit": 0, "completed": starved.completed},
        "ungated_run": {"completed": gated.completed}},
    "lab6_final_brief.md": rt2.state.draft["document"] if rt2.state.draft else "",
    "lab6_acceptance_checks.json": [{"check": n, "passed": ok, "evidence": e}
                                    for n, ok, e in checks],
}
for fname, obj in artifacts.items():
    (outdir / fname).write_text(obj if isinstance(obj, str)
                                else json.dumps(obj, indent=2, default=str))

print("=" * 86)
print("ARTIFACTS WRITTEN")
print("=" * 86)
for fname in artifacts:
    print(f"  {outdir / fname}  ({(outdir / fname).stat().st_size:,} bytes)")
print(f"\n  Checkpoint file     : {checkpoints.path(RUN_ID)}")
print(f"  Actions in the plan : {len(PLAN)}")
print(f"  Audit entries       : {len(rt2.audit)}")
print(f"  Acceptance checks   : {sum(ok for _, ok, _ in checks)}/{len(checks)} passed")


ARTIFACTS WRITTEN
  lab6_artifacts/lab6_run_plan.json  (1,423 bytes)
  lab6_artifacts/lab6_audit_log.json  (9,590 bytes)
  lab6_artifacts/lab6_checkpoint.json  (14,390 bytes)
  lab6_artifacts/lab6_crash_and_resume.json  (580 bytes)
  lab6_artifacts/lab6_budget_refusals.json  (355 bytes)
  lab6_artifacts/lab6_final_brief.md  (924 bytes)
  lab6_artifacts/lab6_acceptance_checks.json  (2,209 bytes)

  Checkpoint file     : lab6_checkpoints/EDSS-RUNTIME-DEMO.checkpoint.json
  Actions in the plan : 7
  Audit entries       : 14
  Acceptance checks   : 13/13 passed


---

## Where Lab 6 leaves the worklet

**Nodes activated:** the Coordinator runtime around every node.

**What exists now:** a run that cannot overspend, cannot skip a dependency, cannot publish without a named human, cannot lose work to a crash, and cannot do anything that is not in the log.

**What Lab 7 adds:** the graph. The run plan in Cell 4 is a *list* with dependency hints. Lab 7 makes the wiring explicit — parallel branches, conditional edges, and a red back-edge from validation to regeneration — so the pipeline stops being a sequence and becomes a node graph.

> Lab 7 acceptance (Chapter 15): the worklet moves from requirement to release through **explicit stages**.